# Retrieval Demo — Truy xuất điều khoản từ ChromaDB

Notebook này gồm 2 phần:
- **Phần 1:** Báo cáo cơ sở dữ liệu (Index Report) — thống kê những gì đã được lưu trong ChromaDB
- **Phần 2:** Demo truy vấn (Retrieval Demo) — tìm kiếm ngữ nghĩa và lọc theo điều khoản cụ thể

> **Yêu cầu:** Đã chạy xong `Ingestion_pipeline_test.ipynb` để có dữ liệu trong ChromaDB và thư mục `chroma_db/`.

---
## Cell 1 — Khởi tạo: Import thư viện và kết nối ChromaDB

In [1]:
# === Cell 1: Import + Kết nối ChromaDB + Load embedding model ===
from pathlib import Path
from collections import Counter
import os
import json

import chromadb
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from pyvi import ViTokenizer

# --- Paths ---
ROOT      = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CHROMA_DIR = ROOT / 'chroma_db'

assert CHROMA_DIR.exists(), (
    f"[FAIL] Không tìm thấy thư mục ChromaDB tại: {CHROMA_DIR}\n"
    "→ Hãy chạy Ingestion_pipeline_test.ipynb trước!"
)

# --- Kết nối ChromaDB ---
client     = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = client.get_or_create_collection('legal_chunks')
total      = collection.count()

print(f"[OK] Kết nối ChromaDB thành công")
print(f"     Collection : legal_chunks")
print(f"     Tổng chunks: {total}")
print(f"     Chroma path: {CHROMA_DIR}")

# --- Load embedding model (để dùng ở Phần 2) ---
load_dotenv(ROOT / '.env')
HF_TOKEN = os.getenv('HF_TOKEN')

print("\n[...] Đang tải embedding model (lần đầu ~20s)...")
embedder = SentenceTransformer('huyydangg/DEk21_hcmute_embedding', token=HF_TOKEN)
print("[OK] Embedding model sẵn sàng")

d:\PTIT\TTCS\PRJ\Rag-Based-Legal-Comparison\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[OK] Kết nối ChromaDB thành công
     Collection : legal_chunks
     Tổng chunks: 559
     Chroma path: d:\PTIT\TTCS\PRJ\Rag-Based-Legal-Comparison\chroma_db

[...] Đang tải embedding model (lần đầu ~20s)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8229.80it/s]


[OK] Embedding model sẵn sàng


---
## Phần 1 — Báo cáo Cơ sở dữ liệu (Index Report)

In [2]:
# === Cell 2: Thống kê tổng quan ChromaDB ===

all_data = collection.get(include=['metadatas', 'documents'])
all_meta = all_data['metadatas']
all_docs = all_data['documents']

# --- Đếm phân bố ---
doc_counts   = Counter(m['doc_id']     for m in all_meta)
type_counts  = Counter(m['chunk_type'] for m in all_meta)
chuong_dist  = Counter(
    f"Chương {m['chuong_so']}" if m['chuong_so'] != '0' else '(header)'
    for m in all_meta
)
char_lens = [len(d) for d in all_docs]

# --- In báo cáo ---
SEP = "═" * 65
print(SEP)
print("  BÁO CÁO CƠ SỞ DỮ LIỆU — ChromaDB  (collection: legal_chunks)")
print(SEP)
print(f"  Tổng số chunks lưu trong DB : {total}")
print(f"  Số văn bản luật             : {len(doc_counts)}")
print()

print("── 1. Số chunks theo từng văn bản luật (doc_id) ──")
print(f"  {'Văn bản (doc_id)':<40} {'Số chunks':>10}")
print(f"  {'-'*40} {'-'*10}")
for doc, cnt in sorted(doc_counts.items()):
    print(f"  {doc:<40} {cnt:>10}")
print()

print("── 2. Số chunks theo loại (chunk_type) ──")
for t, cnt in sorted(type_counts.items(), key=lambda x: -x[1]):
    bar = '█' * (cnt // 10)
    print(f"  {t:<15} {cnt:>5}  {bar}")
print()

print("── 3. Phân bố theo Chương (top 10) ──")
for label, cnt in sorted(chuong_dist.items(), key=lambda x: -x[1])[:10]:
    bar = '█' * (cnt // 5)
    print(f"  {label:<15} {cnt:>5}  {bar}")
print()

print("── 4. Thống kê độ dài chunk (số ký tự) ──")
print(f"  Ngắn nhất : {min(char_lens):>6} ký tự")
print(f"  Dài nhất  : {max(char_lens):>6} ký tự")
print(f"  Trung bình: {sum(char_lens)//len(char_lens):>6} ký tự")
print(SEP)

═════════════════════════════════════════════════════════════════
  BÁO CÁO CƠ SỞ DỮ LIỆU — ChromaDB  (collection: legal_chunks)
═════════════════════════════════════════════════════════════════
  Tổng số chunks lưu trong DB : 559
  Số văn bản luật             : 8

── 1. Số chunks theo từng văn bản luật (doc_id) ──
  Văn bản (doc_id)                          Số chunks
  ---------------------------------------- ----------
  112_2025_QH15_586814                             59
  114_2025_QH15_658530                             47
  116_2025_QH15_666020                             46
  127_2025_QH15_686325                            181
  133_2025_QH15_675211                             28
  135_2025_QH15_675213                             96
  143_2025_QH15_681550                             53
  148_2025_QH15_675262                             49

── 2. Số chunks theo loại (chunk_type) ──
  article           543  ██████████████████████████████████████████████████████
  header            

In [3]:
# === Cell 3: Bảng chi tiết — số Điều trong từng Chương của mỗi văn bản ===

# Nhóm theo (doc_id, chuong_so) → đếm số điều
from collections import defaultdict

doc_chuong = defaultdict(lambda: defaultdict(int))
for m in all_meta:
    if m['chunk_type'] == 'article':
        doc_chuong[m['doc_id']][m['chuong_so']] += 1

print("BẢN ĐỒ CẤU TRÚC: Số Điều mỗi Chương trong từng văn bản luật")
print()

for doc_id, chuong_map in sorted(doc_chuong.items()):
    total_dieu = sum(chuong_map.values())
    print(f"📄 {doc_id}  (tổng: {total_dieu} điều)")
    for chuong_so, cnt in sorted(chuong_map.items(), key=lambda x: x[0]):
        label = f"Chương {chuong_so}" if chuong_so != '0' else "(chưa vào chương)"
        bar   = '▪' * cnt
        print(f"   {label:<15}  {cnt:>3} điều  {bar}")
    print()

BẢN ĐỒ CẤU TRÚC: Số Điều mỗi Chương trong từng văn bản luật

📄 112_2025_QH15_586814  (tổng: 57 điều)
   Chương I          15 điều  ▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪
   Chương II         15 điều  ▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪
   Chương III         9 điều  ▪▪▪▪▪▪▪▪▪
   Chương IV          8 điều  ▪▪▪▪▪▪▪▪
   Chương V           6 điều  ▪▪▪▪▪▪
   Chương VI          4 điều  ▪▪▪▪

📄 114_2025_QH15_658530  (tổng: 45 điều)
   Chương I          13 điều  ▪▪▪▪▪▪▪▪▪▪▪▪▪
   Chương II         12 điều  ▪▪▪▪▪▪▪▪▪▪▪▪
   Chương III         7 điều  ▪▪▪▪▪▪▪
   Chương IV          4 điều  ▪▪▪▪
   Chương V           6 điều  ▪▪▪▪▪▪
   Chương VI          3 điều  ▪▪▪

📄 116_2025_QH15_666020  (tổng: 44 điều)
   Chương I           6 điều  ▪▪▪▪▪▪
   Chương II          5 điều  ▪▪▪▪▪
   Chương III        10 điều  ▪▪▪▪▪▪▪▪▪▪
   Chương IV          4 điều  ▪▪▪▪
   Chương V           3 điều  ▪▪▪
   Chương VI          9 điều  ▪▪▪▪▪▪▪▪▪
   Chương VII         4 điều  ▪▪▪▪
   Chương VIII        3 điều  ▪▪▪

📄 127_2025_QH15_686325  (tổng: 179 điều)
   Chương

---
## Phần 2 — Demo Truy vấn (Retrieval Demo)

In [11]:
# === Cell 4: Hàm trợ giúp hiển thị kết quả đẹp ===

def show_chunk(rank, chunk_id, meta, document, distance=None):
    """In thông tin một chunk kết quả ra màn hình dễ đọc."""
    SEP = "─" * 65
    chuong_label = f"Chương {meta['chuong_so']}" if meta['chuong_so'] != '0' else "(Header)"
    muc_label    = f"Mục {meta['muc_so']}"       if meta['muc_so']    != '0' else "(Không có Mục)"
    score_str    = f"  |  Do lien quan: {1 / (1 + distance):.4f}" if distance is not None else ""

    print(SEP)
    print(f"  [{rank}] {meta['clause_id']} — {meta['article_title'] or '(không có tiêu đề)'}")
    print(f"       Loại     : {meta['chunk_type']}{score_str}")
    print(f"       Nguồn    : {meta['doc_id']}")
    print(f"       Vị trí   : {chuong_label}  /  {muc_label}")
    print(f"       Khoản/Điểm: {meta['khoan_count']} khoản,  {meta['diem_count']} điểm")
    print(f"       Chunk ID : {chunk_id}")
    print(f"       Nội dung :")
    preview = document[:400].strip()
    for line in preview.split('\n')[:8]:    # tối đa 8 dòng đầu
        print(f"         {line}")
    if len(document) > 400:
        print(f"         ... [{len(document)} ký tự tổng cộng]")

print("[OK] Hàm show_chunk() đã sẵn sàng")

[OK] Hàm show_chunk() đã sẵn sàng


In [12]:
# === Cell 5: Demo 1 — Truy vấn ngữ nghĩa (Semantic Search) ===
#
# Người dùng nhập câu hỏi tiếng Việt tự nhiên.
# Hệ thống embed câu hỏi → tìm k chunks có vector gần nhất trong ChromaDB.
#
# ✏️  Thay đổi câu hỏi ở đây để thử nghiệm:
QUERY   = "điều kiện để được cấp phép xây dựng"
TOP_K   = 5      # Số kết quả muốn lấy

# --- Embed câu hỏi (có word-segmentation tiếng Việt) ---
query_seg = ViTokenizer.tokenize(QUERY)
query_vec = embedder.encode([query_seg], convert_to_numpy=True)

# --- Query ChromaDB ---
results = collection.query(
    query_embeddings=query_vec.tolist(),
    n_results=TOP_K,
    include=['metadatas', 'documents', 'distances']
)

# --- Hiển thị ---
print(f"\n🔍 CÂU TRUY VẤN NGỮ NGHĨA: \"{QUERY}\"")
print(f"   (Segmented: {query_seg[:80]}...)")
print(f"   → Top {TOP_K} kết quả:\n")

ids       = results['ids'][0]
metas     = results['metadatas'][0]
docs      = results['documents'][0]
distances = results['distances'][0]

for rank, (cid, meta, doc, dist) in enumerate(zip(ids, metas, docs, distances), start=1):
    show_chunk(rank, cid, meta, doc, distance=dist)
print("─" * 65)


🔍 CÂU TRUY VẤN NGỮ NGHĨA: "điều kiện để được cấp phép xây dựng"
   (Segmented: điều_kiện để được cấp phép xây_dựng...)
   → Top 5 kết quả:

─────────────────────────────────────────────────────────────────
  [1] Dieu_43 — Quy định chung về cấp giấy phép xây dựng
       Loại     : article  |  Do lien quan: 0.0393
       Nguồn    : 135_2025_QH15_675213
       Vị trí   : Chương III  /  (Không có Mục)
       Khoản/Điểm: 3 khoản,  13 điểm
       Chunk ID : 135_2025_QH15_675213_v1_0043
       Nội dung :
         Điều 43. Quy định chung về cấp giấy phép xây dựng
         1. Giấy phép xây dựng bao gồm các loại sau đây:
         a) Giấy phép xây dựng mới;
         b) Giấy phép sửa chữa, cải tạo, di dời công trình;
         c) Giấy phép xây dựng có thời hạn.
         2. Trước khi khởi công xây dựng công trình, chủ đầu tư phải có giấy phép xây dựng trừ các trường hợp sau đây:
         a) Công trình bí mật nhà nước; công trình xây dựng khẩn cấp, cấp bách; công trì
         ... [3929 ký tự tổng cộ

In [13]:
# === Cell 6: Demo 1b — Thử nhiều câu hỏi để so sánh độ chính xác ===

SAMPLE_QUERIES = [
    "quyền và nghĩa vụ của người sử dụng đất",
    "xử phạt vi phạm hành chính trong lĩnh vực môi trường",
    "điều kiện thành lập doanh nghiệp",
    "trách nhiệm của cơ quan nhà nước",
]

print("🔍 KIỂM TRA NHANH — Top 1 kết quả cho mỗi câu hỏi mẫu")
print("=" * 65)

for q in SAMPLE_QUERIES:
    q_seg = ViTokenizer.tokenize(q)
    q_vec = embedder.encode([q_seg], convert_to_numpy=True)
    res   = collection.query(
        query_embeddings=q_vec.tolist(),
        n_results=1,
        include=['metadatas', 'distances']
    )
    m    = res['metadatas'][0][0]
    dist = res['distances'][0][0]
    score = 1 / (1 + dist)
    print(f"  ❓ {q}")
    print(f"     → {m['clause_id']} | {m['article_title'][:50]}... | do_lien_quan={score:.3f} | nguồn: {m['doc_id']}")
    print()

🔍 KIỂM TRA NHANH — Top 1 kết quả cho mỗi câu hỏi mẫu
  ❓ quyền và nghĩa vụ của người sử dụng đất
     → Dieu_40 | Quyền, nghĩa vụ và trách nhiệm của chủ đầu tư... | do_lien_quan=0.019 | nguồn: 135_2025_QH15_675213

  ❓ xử phạt vi phạm hành chính trong lĩnh vực môi trường
     → Dieu_114 | Xử lý người chấp hành án phạt quản chế vi phạm ngh... | do_lien_quan=0.018 | nguồn: 127_2025_QH15_686325

  ❓ điều kiện thành lập doanh nghiệp
     → Dieu_19 | Đầu tư thành lập tổ chức kinh tế... | do_lien_quan=0.025 | nguồn: 143_2025_QH15_681550

  ❓ trách nhiệm của cơ quan nhà nước
     → Dieu_32 | Cung cấp dịch vụ công trực tuyến... | do_lien_quan=0.024 | nguồn: 148_2025_QH15_675262



In [7]:
# === Cell 7: Demo 2 — Lọc theo Điều khoản cụ thể (Exact/Filter Lookup) ===
#
# Không cần AI, chỉ dùng metadata filter để lấy đúng điều muốn xem.
#
# ✏️  Thay đổi các tham số dưới đây:
FILTER_ARTICLE_NUMBER = "5"   # Điều số bao nhiêu? (để None nếu không lọc theo Điều)
FILTER_DOC_ID         = None  # Tên file luật cụ thể? (để None để tìm trong tất cả)

# --- Xây dựng điều kiện lọc ---
where_clause = {"article_number": FILTER_ARTICLE_NUMBER}
if FILTER_DOC_ID:
    where_clause = {"$and": [
        {"article_number": FILTER_ARTICLE_NUMBER},
        {"doc_id": FILTER_DOC_ID}
    ]}

result = collection.get(
    where=where_clause,
    include=['metadatas', 'documents']
)

ids   = result['ids']
metas = result['metadatas']
docs  = result['documents']

print(f"\n🔎 LỌC THEO ĐIỀU KHOẢN CỤ THỂ")
print(f"   Điều số    : {FILTER_ARTICLE_NUMBER}")
print(f"   Văn bản    : {FILTER_DOC_ID or '(tất cả)'}")
print(f"   → Tìm thấy: {len(ids)} kết quả\n")

for rank, (cid, meta, doc) in enumerate(zip(ids, metas, docs), start=1):
    show_chunk(rank, cid, meta, doc)

if not ids:
    print("  ⚠️  Không tìm thấy điều khoản nào phù hợp.")
print("─" * 65)


🔎 LỌC THEO ĐIỀU KHOẢN CỤ THỂ
   Điều số    : 5
   Văn bản    : (tất cả)
   → Tìm thấy: 8 kết quả

─────────────────────────────────────────────────────────────────
  [1] Dieu_5 — Hệ thống quy hoạch
       Loại     : article
       Nguồn    : 112_2025_QH15_586814
       Vị trí   : Chương I  /  (Không có Mục)
       Khoản/Điểm: 3 khoản,  17 điểm
       Chunk ID : 112_2025_QH15_586814_v1_0005
       Nội dung :
         Điều 5. Hệ thống quy hoạch
         1. Hệ thống quy hoạch bao gồm:
         a) Quy hoạch cấp quốc gia, gồm: quy hoạch tổng thể quốc gia, quy hoạch không gian biển quốc gia, quy hoạch sử dụng đất quốc gia, quy hoạch ngành;
         b) Quy hoạch vùng. Chính phủ xác định các vùng cần lập quy hoạch;
         c) Quy hoạch tỉnh;
         d) Quy hoạch chi tiết ngành;
         đ) Quy hoạch đô thị và nông thôn;
         e) Quy hoạch đơn vị hành chính - kinh tế đặc biệ
         ... [3731 ký tự tổng cộng]
─────────────────────────────────────────────────────────────────
  [2] Dieu_5 

In [8]:
# === Cell 8: Demo 2b — Lọc tất cả Điều trong một Chương cụ thể ===
#
# ✏️  Thay đổi tham số:
FILTER_CHUONG = "I"    # Chương nào? (số La Mã: 'I', 'II', 'III', ...)
FILTER_DOC    = None   # Giới hạn trong 1 văn bản? (None = tất cả)

# --- Xây dựng filter ---
where = {"$and": [
    {"chuong_so":   FILTER_CHUONG},
    {"chunk_type":  "article"}
]}
if FILTER_DOC:
    where["$and"].append({"doc_id": FILTER_DOC})

result = collection.get(where=where, include=['metadatas', 'documents'])

ids   = result['ids']
metas = result['metadatas']
docs  = result['documents']

doc_filter_str = FILTER_DOC or '(tất cả văn bản)'
print(f"\n📂 TẤT CẢ ĐIỀU TRONG CHƯƠNG {FILTER_CHUONG} — {doc_filter_str}")
print(f"   → Tìm thấy {len(ids)} điều\n")

# In tóm tắt dạng bảng
print(f"  {'STT':>4}  {'Điều':>8}  {'Tiêu đề':<45}  {'Văn bản'}")
print(f"  {'─'*4}  {'─'*8}  {'─'*45}  {'─'*30}")
for i, (cid, meta) in enumerate(zip(ids, metas), start=1):
    title  = (meta['article_title'] or '(không có)')[:45]
    doc_id = meta['doc_id'][:30]
    print(f"  {i:>4}  {meta['clause_id']:>8}  {title:<45}  {doc_id}")

print()

# Xem chi tiết điều đầu tiên
if ids:
    print("--- Xem chi tiết điều đầu tiên trong danh sách ---")
    show_chunk(1, ids[0], metas[0], docs[0])
    print("─" * 65)


📂 TẤT CẢ ĐIỀU TRONG CHƯƠNG I — (tất cả văn bản)
   → Tìm thấy 82 điều

   STT      Điều  Tiêu đề                                        Văn bản
  ────  ────────  ─────────────────────────────────────────────  ──────────────────────────────
     1    Dieu_1  Phạm vi điều chỉnh                             112_2025_QH15_586814
     2    Dieu_2  Đối tượng áp dụng                              112_2025_QH15_586814
     3    Dieu_4  Nguyên tắc cơ bản trong hoạt động quy hoạch    112_2025_QH15_586814
     4    Dieu_5  Hệ thống quy hoạch                             112_2025_QH15_586814
     5    Dieu_6  Nguyên tắc xác định quy hoạch phải điều chỉnh  112_2025_QH15_586814
     6    Dieu_7  Thời kỳ, tầm nhìn quy hoạch                    112_2025_QH15_586814
     7    Dieu_8  Sơ đồ quy hoạch, bản đồ quy hoạch              112_2025_QH15_586814
     8    Dieu_9  Trình tự lập, thẩm định, quyết định hoặc phê   112_2025_QH15_586814
     9   Dieu_10  Chi phí cho hoạt động quy hoạch                112_20

In [9]:
# === Cell 9: Demo 3 — Xem toàn bộ nội dung một Điều cụ thể ===
#
# ✏️  Thay đổi:
VIEW_ARTICLE_NUMBER = "1"                        # Điều số bao nhiêu?
VIEW_DOC_ID         = None                       # None = lấy từ văn bản đầu tiên tìm được

result = collection.get(
    where={"article_number": VIEW_ARTICLE_NUMBER},
    include=['metadatas', 'documents']
)

if not result['ids']:
    print(f"⚠️  Không tìm thấy Điều {VIEW_ARTICLE_NUMBER}")
else:
    # Nếu có nhiều kết quả (cùng số điều ở nhiều luật), lấy kết quả đầu tiên
    idx  = 0
    meta = result['metadatas'][idx]
    doc  = result['documents'][idx]
    cid  = result['ids'][idx]

    chuong_label = f"Chương {meta['chuong_so']}" if meta['chuong_so'] != '0' else '(Header)'
    muc_label    = f"Mục {meta['muc_so']}"       if meta['muc_so']    != '0' else ''

    print(f"\n{'═'*65}")
    print(f"  NỘI DUNG ĐẦY ĐỦ — {meta['clause_id']}")
    print(f"  Tiêu đề : {meta['article_title'] or '(không có)'}")
    print(f"  Nguồn   : {meta['doc_id']}")
    print(f"  Vị trí  : {chuong_label}  {muc_label}")
    print(f"  Loại    : {meta['chunk_type']}   |   {meta['khoan_count']} khoản  /  {meta['diem_count']} điểm")
    print(f"{'─'*65}")
    print(doc)
    print(f"{'═'*65}")
    if len(result['ids']) > 1:
        print(f"\n  💡 Lưu ý: Tìm thấy {len(result['ids'])} văn bản có Điều {VIEW_ARTICLE_NUMBER}:")
        for m in result['metadatas']:
            print(f"     - {m['doc_id']}: {m['article_title']}")


═════════════════════════════════════════════════════════════════
  NỘI DUNG ĐẦY ĐỦ — Dieu_1
  Tiêu đề : Phạm vi điều chỉnh
  Nguồn   : 112_2025_QH15_586814
  Vị trí  : Chương I  
  Loại    : article   |   0 khoản  /  0 điểm
─────────────────────────────────────────────────────────────────
Điều 1. Phạm vi điều chỉnh
Luật này quy định hệ thống quy hoạch; việc lập, thẩm định, quyết định hoặc phê duyệt, công bố, cung cấp thông tin, thực hiện, đánh giá và điều chỉnh quy hoạch; quản lý nhà nước về hoạt động quy hoạch.
═════════════════════════════════════════════════════════════════

  💡 Lưu ý: Tìm thấy 8 văn bản có Điều 1:
     - 112_2025_QH15_586814: Phạm vi điều chỉnh
     - 114_2025_QH15_658530: Phạm vi điều chỉnh, đối tượng áp dụng
     - 116_2025_QH15_666020: Phạm vi điều chỉnh và đối tượng áp dụng
     - 127_2025_QH15_686325: Phạm vi điều chỉnh
     - 133_2025_QH15_675211: Phạm vi điều chỉnh
     - 135_2025_QH15_675213: Phạm vi điều chỉnh
     - 143_2025_QH15_681550: Phạm vi điều ch

In [14]:
# === Cell 10: Demo 4 — Tìm kiếm ngữ nghĩa CÓ lọc metadata (Hybrid Search) ===
#
# Kết hợp: tìm ngữ nghĩa + giới hạn trong 1 văn bản/chương cụ thể.
# Ví dụ: "Tìm các điều nói về quyền hạn" nhưng CHỈ trong Luật Quy Hoạch.
#
# ✏️  Thay đổi:
HYBRID_QUERY   = "quyền hạn và trách nhiệm của cơ quan nhà nước"
HYBRID_DOC_ID  = "112_2025_QH15_586814"   # Giới hạn trong văn bản này (None = tất cả)
HYBRID_TOP_K   = 3

# --- Embed câu hỏi ---
q_seg = ViTokenizer.tokenize(HYBRID_QUERY)
q_vec = embedder.encode([q_seg], convert_to_numpy=True)

# --- Query với filter ---
query_kwargs = dict(
    query_embeddings=q_vec.tolist(),
    n_results=HYBRID_TOP_K,
    include=['metadatas', 'documents', 'distances']
)
if HYBRID_DOC_ID:
    query_kwargs['where'] = {"doc_id": HYBRID_DOC_ID}

results = collection.query(**query_kwargs)

# --- Hiển thị ---
doc_filter_str = HYBRID_DOC_ID or '(tất cả văn bản)'
print(f"\n🔍 HYBRID SEARCH")
print(f"   Câu hỏi   : \"{HYBRID_QUERY}\"")
print(f"   Giới hạn  : {doc_filter_str}")
print(f"   → Top {HYBRID_TOP_K} kết quả:\n")

for rank, (cid, meta, doc, dist) in enumerate(
    zip(results['ids'][0], results['metadatas'][0],
        results['documents'][0], results['distances'][0]),
    start=1
):
    show_chunk(rank, cid, meta, doc, distance=dist)
print("─" * 65)


🔍 HYBRID SEARCH
   Câu hỏi   : "quyền hạn và trách nhiệm của cơ quan nhà nước"
   Giới hạn  : 112_2025_QH15_586814
   → Top 3 kết quả:

─────────────────────────────────────────────────────────────────
  [1] Dieu_12 — Quản lý nhà nước về quy hoạch
       Loại     : article  |  Do lien quan: 0.0202
       Nguồn    : 112_2025_QH15_586814
       Vị trí   : Chương I  /  (Không có Mục)
       Khoản/Điểm: 3 khoản,  0 điểm
       Chunk ID : 112_2025_QH15_586814_v1_0012
       Nội dung :
         Điều 12. Quản lý nhà nước về quy hoạch
         1. Chính phủ thống nhất quản lý nhà nước về quy hoạch trong phạm vi cả nước.
         2. Bộ Tài chính là cơ quan đầu mối giúp Chính phủ thực hiện quản lý nhà nước về quy hoạch.
         3. Các Bộ, Ủy ban nhân dân các cấp thực hiện quản lý nhà nước về quy hoạch trong phạm vi nhiệm vụ, quyền hạn được giao.
─────────────────────────────────────────────────────────────────
  [2] Dieu_17 — Thẩm quyền tổ chức lập quy hoạch
       Loại     : article  |  Do lie